## Setup and Imports
Prepare the notebook environment, patch Windows text loading for UTF-8 template files, and import the libraries used for dataset loading, 4-bit model loading, LoRA training, merging, and inference.

In [ ]:
from pathlib import Path

if not hasattr(Path, "_original_read_text"):
    Path._original_read_text = Path.read_text

    def read_text_utf8(self, encoding=None, errors=None):
        return Path._original_read_text(
            self,
            encoding=encoding or "utf-8",
            errors=errors,
        )

    Path.read_text = read_text_utf8

import torch
from collections import Counter
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

## Dataset and CUDA Check
Confirm that CUDA is visible to PyTorch, then load the local JSONL training set and print its shape so the training run starts from the expected 15,000 examples.

In [9]:
cuda_available = torch.cuda.is_available()
print("CUDA available:", cuda_available)

if cuda_available:
    print("GPU:", torch.cuda.get_device_name(0))

dataset = load_dataset("json", data_files="datasets/golden_train.jsonl", split="train")

print("Rows:", len(dataset))
print("Columns:", dataset.column_names)

CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Rows: 15000
Columns: ['text']


## Base Model and Tokenizer
Load Qwen/Qwen2.5-Coder-1.5B-Instruct in 4-bit precision with bfloat16 compute, place it automatically on the available device, and align the tokenizer padding token with EOS for causal language modeling.

In [10]:
model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Model and tokenizer loaded successfully.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model and tokenizer loaded successfully.


## LoRA Adapter Setup
Prepare the quantized model for k-bit training, attach lightweight LoRA adapters to the attention projection layers, and print the trainable parameter summary to verify that only the adapter weights will be updated.

In [11]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)

trainable_dtypes = Counter(
    str(param.dtype)
    for param in model.parameters()
    if param.requires_grad
)

print(trainable_dtypes)

model.print_trainable_parameters()

Counter({'torch.float32': 224})
trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


## Training Configuration
Run a 6GB VRAM-friendly 300-step SFT pass with batch size 1, gradient accumulation, checkpointed activations, bfloat16 training, a conservative gradient norm, and the prepared text column as the training source.

In [5]:
model.config.use_cache = False

training_args = SFTConfig(
    output_dir="./mini-gpt-coder-FULL",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    fp16=False,
    bf16=True,
    learning_rate=2e-4,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    max_steps=300,
    logging_steps=10,
    dataset_text_field="text",
    max_length=2048,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
10,2.124152
20,1.853761
30,1.665174
40,1.521502
50,1.398782
60,1.340503
70,1.358431
80,1.392592
90,1.351151
100,1.319720


TrainOutput(global_step=300, training_loss=1.3717543601989746, metrics={'train_runtime': 480.5136, 'train_samples_per_second': 2.497, 'train_steps_per_second': 0.624, 'total_flos': 1658176987806720.0, 'train_loss': 1.3717543601989746})

## Save LoRA Adapters
Save the trained adapter weights and tokenizer separately so the fine-tuned LoRA checkpoint can be reused, merged, or inspected before producing the standalone model.

In [6]:
# Save the LoRA adapters and the tokenizer locally
trainer.model.save_pretrained("./mini-gpt-coder-adapter")
tokenizer.save_pretrained("./mini-gpt-coder-adapter")

print("Adapters saved successfully!")

 Adapters saved successfully!


## Merge and Export Standalone Model
Reload the base model in fp16, merge the trained LoRA adapters into it, and save a self-contained model directory that no longer needs PEFT adapter loading for inference.

In [ ]:
print("Merging LoRA adapters into base model...")

# Reload base model in fp16 for merging (can't merge in 4-bit)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Load and merge the LoRA adapters
merged_model = PeftModel.from_pretrained(base_model, "./mini-gpt-coder-adapter")
merged_model = merged_model.merge_and_unload()

# Save the fully merged standalone model
merged_model.save_pretrained("./mini-gpt-coder-merged")
tokenizer.save_pretrained("./mini-gpt-coder-merged")

print("=" * 80)
print("MERGED MODEL EXPORT COMPLETE!")
print("Standalone model saved to: ./mini-gpt-coder-merged")
print("No PEFT or adapter files needed for inference.")
print("=" * 80)

Merging LoRA adapters into base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MERGED MODEL EXPORT COMPLETE!
Standalone model saved to: ./mini-gpt-coder-merged
No PEFT or adapter files needed for inference.


## Inference Smoke Test
Load the merged model as a plain causal language model, format a prompt with the same chat markers used during training, generate a deterministic response, strip leftover tags, and print the resulting Python code.

In [13]:
model = AutoModelForCausalLM.from_pretrained(
    "./mini-gpt-coder-merged",
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("./mini-gpt-coder-merged")
tokenizer.pad_token = tokenizer.eos_token
model.eval()

user_prompt = "Write a Python function to read a CSV file and return the headers."

prompt = (
    "<|im_start|>user\n"
    "Write a Python function for the following:\n"
    f"{user_prompt}<|im_end|>\n"
    "<|im_start|>assistant\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

stop_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_id,
    )

decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=False)

output = decoded.replace(prompt, "")
output = output.split("<|im_end|>")[0]
output = output.replace("<|im_start|>assistant", "")
output = output.strip()

print(output)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

def get_headers(csv_file):
    """Reads a CSV file and returns the headers."""
    with open(csv_file, 'r') as f:
        reader = csv.reader(f)
        header_row = next(reader)

    # Remove any empty strings from the header row.
    header_row = [h.strip() for h in header_row if h]

    return header_row
